In [ ]:

# =============================================================================
# Head-to-head over many seeds, BOTH test sets, TRUE robust pipeline.
#   arm A  classifier : ConvNeXt-Large trained on unmasked -> eval unmasked
#   arm B  pipeline   : Swin-Tiny trained on oracle-masked -> eval on PREDICTED masks
#                       (localiser conf 0.05 + full-frame fallback, precomputed)
# Goal: replace the 3-seed paired CI (external [-3.83,+10.31]) with something tight.
# =============================================================================
import subprocess, sys, os, json, time, random, glob
gpu=""
try: gpu=subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                        capture_output=True,text=True,timeout=60).stdout.strip()
except Exception as e: print("nvidia-smi:",e)
print("GPU:",gpu,flush=True)
if "P100" in gpu:
    print("P100 -> cu121 torch build",flush=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","torch==2.5.1","torchvision==0.20.1",
                    "--index-url","https://download.pytorch.org/whl/cu121"],check=False)
subprocess.run([sys.executable,"-m","pip","install","-q","timm==1.0.11"],check=False)

import numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
print("torch",torch.__version__,"cuda",torch.cuda.is_available(),flush=True)
if torch.cuda.is_available():
    print("cap",torch.cuda.get_device_capability(0)); torch.zeros(4,device="cuda"); print("smoke OK",flush=True)

def find(pred):
    for d,subs,_ in os.walk("/kaggle/input"):
        if pred(set(subs)): return d
    return None
ROOT4 = find(lambda s: {"unmasked","masked"} <= s)
EVAL  = find(lambda s: {"internal_predmask","external_unmasked","external_predmask"} <= s)
print("ROOT4:",ROOT4); print("EVAL :",EVAL,flush=True)
assert ROOT4 and EVAL

DEV="cuda" if torch.cuda.is_available() else "cpu"
SEEDS=[0,1,2,3,4,5,6,7,8,9]
EPOCHS,BS,LR,IMG=12,16,1e-4,224
NORM=([0.485,0.456,0.406],[0.229,0.224,0.225])
tf_tr=transforms.Compose([transforms.Resize((IMG,IMG)),transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2),transforms.RandomRotation(10),
    transforms.ToTensor(),transforms.Normalize(*NORM)])
tf_ev=transforms.Compose([transforms.Resize((IMG,IMG)),transforms.ToTensor(),transforms.Normalize(*NORM)])

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def dl(path,tf,shuffle=False,seed=0):
    ds=datasets.ImageFolder(path,tf)
    g=torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds,batch_size=BS,shuffle=shuffle,num_workers=2,generator=g if shuffle else None),ds

def cw(ds):
    c=np.bincount([y for _,y in ds.samples],minlength=3).astype(float)
    return torch.tensor(c.sum()/(3*np.maximum(c,1)),dtype=torch.float32,device=DEV)

@torch.no_grad()
def ev(model,loader):
    model.eval(); P=[];Y=[]
    for x,y in loader:
        P+=list(model(x.to(DEV)).argmax(1).cpu().numpy()); Y+=list(y.numpy())
    P=np.array(P);Y=np.array(Y)
    return (float(100*(P==Y).mean()),
            float(100*np.mean([(P[Y==c]==c).mean() for c in np.unique(Y)])),P.tolist(),Y.tolist())

ARMS=[("A_classifier","convnext_large","unmasked",
       f"{ROOT4}/unmasked/test", f"{EVAL}/external_unmasked"),
      ("B_pipeline","swin_tiny_patch4_window7_224","masked",
       f"{EVAL}/internal_predmask", f"{EVAL}/external_predmask")]

OUT="/kaggle/working/h2h_multiseed.json"; res=[]; t0=time.time()
for seed in SEEDS:
    for name,arch,cond,int_path,ext_path in ARMS:
        try:
            seed_all(seed)
            trl,trds=dl(f"{ROOT4}/{cond}/train",tf_tr,True,seed)
            val,_   =dl(f"{ROOT4}/{cond}/valid",tf_ev)
            iL,_    =dl(int_path,tf_ev)
            eL,_    =dl(ext_path,tf_ev)
            model=timm.create_model(arch,pretrained=True,num_classes=3).to(DEV)
            opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
            sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
            crit=nn.CrossEntropyLoss(weight=cw(trds))
            amp=(DEV=="cuda"); scaler=torch.cuda.amp.GradScaler(enabled=amp)
            bv,bs_=-1,None
            for e in range(EPOCHS):
                model.train()
                for x,y in trl:
                    x,y=x.to(DEV),y.to(DEV); opt.zero_grad(set_to_none=True)
                    with torch.cuda.amp.autocast(enabled=amp): loss=crit(model(x),y)
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                sch.step()
                v,_,_,_=ev(model,val)
                if v>bv: bv=v; bs_={k:t.detach().cpu().clone() for k,t in model.state_dict().items()}
            model.load_state_dict(bs_)
            ia,ib,ip,iy = ev(model,iL)
            ea,eb,ep,ey = ev(model,eL)
            res.append(dict(arm=name,arch=arch,cond=cond,seed=seed,best_val=bv,
                            int_acc=ia,int_bal=ib,ext_acc=ea,ext_bal=eb,
                            int_preds=ip,int_labels=iy,ext_preds=ep,ext_labels=ey))
            print(f"  {name:<13} seed{seed}: INT {ia:.2f} (bal {ib:.2f}) | EXT {ea:.2f} (bal {eb:.2f})"
                  f"  [{(time.time()-t0)/60:.1f}m]",flush=True)
            del model; torch.cuda.empty_cache()
        except Exception as ex:
            print(f"  !! {name} seed{seed}: {type(ex).__name__}: {ex}",flush=True)
        json.dump(res,open(OUT,"w"))
    # paired summary after each complete seed
    A={r["seed"]:r for r in res if r["arm"]=="A_classifier"}
    B={r["seed"]:r for r in res if r["arm"]=="B_pipeline"}
    common=sorted(set(A)&set(B))
    if len(common)>=2:
        di=[A[s]["int_acc"]-B[s]["int_acc"] for s in common]
        de=[A[s]["ext_acc"]-B[s]["ext_acc"] for s in common]
        print(f"  -- paired over {len(common)} seeds: internal {np.mean(di):+.2f}pp  external {np.mean(de):+.2f}pp",flush=True)
print("\nDONE, seeds completed:",sorted({r['seed'] for r in res}))
json.dump(res,open(OUT,"w"))
